# Q3 — Pipeline ETL Completo: NYC Taxi

**Semana 2 | Spark no Databricks**

Pipeline de ponta a ponta usando o dataset público NYC Taxi disponível em `dbfs:/databricks-datasets/nyctaxi/`.

**Arquitetura Medallion aplicada:**
```
Raw CSV (DBFS)
    ↓  [leitura com schema explícito]
Bronze (df_raw)       ← dados brutos, sem transformação
    ↓  [limpeza + enriquecimento]
Silver (df_clean)     ← dados limpos, colunas derivadas
    ↓  [agregação analítica]
Gold (df_por_hora)    ← métricas de negócio prontas
    ↓  [persiste como Delta]
Delta Table (DBFS)
```

> ⚠️ **Boa prática de produção:** nunca use `inferSchema=True` em produção — o Spark lê o arquivo inteiro para inferir tipos. Defina sempre o schema explicitamente com `StructType`.

---
## Etapa 1 — Explorar o dataset (Bronze)

Antes de ler, sempre explore a estrutura de arquivos disponíveis.

In [ ]:
# Explorar a estrutura do dataset NYC Taxi
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/"))

In [ ]:
# Listar os arquivos disponíveis — usaremos o de dezembro 2019 (tamanho gerenciável)
display(dbutils.fs.ls("dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/"))

---
## Etapa 2 — Ler com schema explícito (Bronze → df_raw)

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, IntegerType, DoubleType
)

# IMPORTANTE: Spark lê CSV por POSIÇÃO quando schema explícito é fornecido — não por nome.
# O header=True apenas pula a primeira linha; os nomes vêm do schema, não do cabeçalho.
# O Yellow Taxi 2019 tem 18 colunas — declarar schema incompleto causa mapeamento errado.
#
# Exemplo do problema (schema com 8 cols):
#   posição 6 = RatecodeID  → mapeado como fare_amount  (valor errado)
#   posição 7 = store_and_fwd_flag ("Y"/"N") → mapeado como tip_amount (null — não parseia Double)
#   posição 8 = PULocationID → mapeado como total_amount (valor errado)

schema_taxi = StructType([
    StructField("vendor_id",             StringType(),    True),  # col  1: VendorID
    StructField("pickup_datetime",       TimestampType(), True),  # col  2: tpep_pickup_datetime
    StructField("dropoff_datetime",      TimestampType(), True),  # col  3: tpep_dropoff_datetime
    StructField("passenger_count",       IntegerType(),   True),  # col  4: passenger_count
    StructField("trip_distance",         DoubleType(),    True),  # col  5: trip_distance
    StructField("rate_code_id",          IntegerType(),   True),  # col  6: RatecodeID
    StructField("store_and_fwd_flag",    StringType(),    True),  # col  7: store_and_fwd_flag
    StructField("pu_location_id",        IntegerType(),   True),  # col  8: PULocationID
    StructField("do_location_id",        IntegerType(),   True),  # col  9: DOLocationID
    StructField("payment_type",          IntegerType(),   True),  # col 10: payment_type
    StructField("fare_amount",           DoubleType(),    True),  # col 11: fare_amount ← posição real
    StructField("extra",                 DoubleType(),    True),  # col 12: extra
    StructField("mta_tax",               DoubleType(),    True),  # col 13: mta_tax
    StructField("tip_amount",            DoubleType(),    True),  # col 14: tip_amount ← posição real
    StructField("tolls_amount",          DoubleType(),    True),  # col 15: tolls_amount
    StructField("improvement_surcharge", DoubleType(),    True),  # col 16: improvement_surcharge
    StructField("total_amount",          DoubleType(),    True),  # col 17: total_amount ← posição real
    StructField("congestion_surcharge",  DoubleType(),    True),  # col 18: congestion_surcharge
])

ARQUIVO_TAXI = "dbfs:/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-12.csv.gz"

df_raw = spark.read.csv(
    ARQUIVO_TAXI,
    schema=schema_taxi,
    header=True
)

print(f"[Bronze] Total de registros: {df_raw.count():,}")
df_raw.printSchema()
display(df_raw.limit(5))

---
## Etapa 3 — Limpeza e enriquecimento (Silver → df_clean)

Removemos nulls, valores inválidos e derivamos novas colunas úteis para análise.

In [ ]:
from pyspark.sql import functions as F

df_clean = (
    df_raw
    # --- Remoção de nulls em colunas críticas ---
    .filter(F.col("fare_amount").isNotNull())
    .filter(F.col("trip_distance").isNotNull())
    .filter(F.col("pickup_datetime").isNotNull())
    .filter(F.col("dropoff_datetime").isNotNull())

    # --- Filtros de domínio: valores fisicamente impossíveis ---
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("passenger_count").between(1, 6))

    # --- Colunas derivadas (enriquecimento) ---
    .withColumn(
        "duracao_min",
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60
    )
    .withColumn("hora_pickup",  F.hour("pickup_datetime"))
    .withColumn("dia_semana",   F.dayofweek("pickup_datetime"))  # 1=Dom, 7=Sáb
    .withColumn("mes",          F.month("pickup_datetime"))
    .withColumn(
        "custo_por_km",
        F.when(F.col("trip_distance") > 0,
               F.col("total_amount") / F.col("trip_distance"))
         .otherwise(None)
    )

    # --- Remoção de viagens absurdas (duração) ---
    .filter(F.col("duracao_min").between(1, 180))
)

total_clean = df_clean.count()
total_raw   = df_raw.count()
removidos   = total_raw - total_clean

print(f"[Bronze] Registros brutos  : {total_raw:>10,}")
print(f"[Silver] Após limpeza      : {total_clean:>10,}")
print(f"         Removidos         : {removidos:>10,} ({removidos/total_raw*100:.1f}%)")
display(df_clean.limit(5))

---
## Etapa 4 — Análise analítica (Gold → df_por_hora)

Agregamos as métricas por hora do dia para responder: **qual horário gera mais receita e gorjeta?**

In [ ]:
# Gold: métricas por hora do dia
df_por_hora = (
    df_clean
    .groupBy("hora_pickup")
    .agg(
        F.count("*").alias("qtd_viagens"),
        F.round(F.avg("fare_amount"),   2).alias("tarifa_media"),
        F.round(F.avg("trip_distance"), 2).alias("distancia_media_km"),
        F.round(F.avg("tip_amount"),    2).alias("gorjeta_media"),
        F.round(F.avg("duracao_min"),   1).alias("duracao_media_min"),
        F.round(F.sum("total_amount"),  0).alias("receita_total"),
    )
    .orderBy("hora_pickup")
)
display(df_por_hora)

In [ ]:
# Registrar df_clean como view temporária para uso no %sql abaixo
df_clean.createOrReplaceTempView("taxi_silver")
print("View 'taxi_silver' registrada — pronta para %sql")

In [ ]:
%sql
-- Top 5 horários com maior gorjeta média
-- Requer que df_clean tenha sido registrado como view na célula ANTERIOR (acima)
SELECT
    hora_pickup,
    ROUND(AVG(tip_amount), 2)    AS gorjeta_media,
    COUNT(*)                     AS qtd_viagens,
    ROUND(SUM(total_amount), 0)  AS receita_total
FROM taxi_silver
GROUP BY hora_pickup
ORDER BY gorjeta_media DESC
LIMIT 5

---
## Etapa 5 — Persistir como Delta (Silver layer no DBFS)

In [ ]:
# Setup: garantir que o volume Unity Catalog existe antes de escrever
# Catálogo disponível neste trial: 'workspace' — DBFS root está desabilitado
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.estudos")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.estudos.semana02_tmp")

SILVER_PATH = "/Volumes/workspace/estudos/semana02_tmp/taxi_silver"
print(f"Silver path: {SILVER_PATH}")

In [ ]:
# Salvar particionado por hora_pickup
# partitionBy cria um subdiretório por valor: hora_pickup=0/, hora_pickup=1/, etc.
(
    df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("hora_pickup")
    .save(SILVER_PATH)
)

print(f"Delta table salva em: {SILVER_PATH}")
print("\nEstrutura de partições:")
display(dbutils.fs.ls(SILVER_PATH))

In [ ]:
# Inspecionar o _delta_log — aqui fica o transaction log que garante ACID
# Cada arquivo JSON é uma "versão" da tabela
print("Transaction log (_delta_log):")
display(dbutils.fs.ls(f"{SILVER_PATH}/_delta_log/"))

In [ ]:
# Ler a tabela Delta de volta e verificar integridade
df_delta_check = spark.read.format("delta").load(SILVER_PATH)
print(f"Registros lidos da Delta: {df_delta_check.count():,}")

# NOTA: .rdd.getNumPartitions() não funciona no Serverless (exige SparkContext internamente)
# Alternativa segura: DESCRIBE DETAIL retorna metadados da tabela Delta
spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_PATH}`").select("numFiles", "sizeInBytes").show()

In [ ]:
# Limpeza — remover dados temporários
dbutils.fs.rm(SILVER_PATH, recurse=True)
print(f"Removido: {SILVER_PATH}")

---
## Resumo Q3 — Pipeline ETL NYC Taxi

| Etapa | Operação | Output |
|-------|----------|--------|
| Bronze | `spark.read.csv(schema=...)` | `df_raw` — dados brutos |
| Silver | filtros + `withColumn()` | `df_clean` — dados limpos |
| Gold | `groupBy().agg()` | `df_por_hora` — métricas |
| Persistência | `.write.format("delta").partitionBy()` | Delta table no UC Volume |
| SQL | `createOrReplaceTempView()` + `%sql` | Análise ad-hoc |

**Pontos de atenção aprendidos:**
- `inferSchema=True` → custo de 1 leitura completa — evitar em produção
- **Schema explícito + CSV: Spark lê por POSIÇÃO, não por nome de coluna** — se o schema tiver menos colunas que o CSV, os valores ficarão mapeados nas colunas erradas (resultado: nulls ou valores sem sentido)
- `partitionBy("hora_pickup")` → 24 subdiretórios, bom para queries filtradas por hora
- `_delta_log/` → cada arquivo JSON é uma versão (base do time travel)
- `df_clean.count()` chama uma ação — materializa o plano de execução inteiro

> 🔗 **Para rodar o mesmo pipeline localmente no VS Code**, veja `Q4_taxi_local.py`